# AcqusitionBuilder Example

This notebook demonstrates how to use the `AcqusitionBuilder` from `dynamic_foraging_processing` to produce a typed `AcquisitionCollection` object from a raw dynamic foraging dataset. This object mirrors the structure of the NWB acquisition module and can later be mapped to `pynwb` objects by a downstream writer. It builds on top of the `RawDataLoader` shown in `raw_data_loader_example.ipynb`.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

from dynamic_foraging_processing.process.acquisition import AcqusitionBuilder
from dynamic_foraging_processing.raw_data_loader import RawDataLoader

## Load a dataset

Point `RawDataLoader` at the root directory of a dataset acquisition. The builder consumes the underlying `dataset` object.

In [2]:
# Update this path to point at a real acquisition directory on your machine.
dataset_path = Path(
    r"C:\Users\arjun.sridhar\Downloads\821586_2026-04-28T200347Z\821586_2026-04-28T200347Z"
)

loader = RawDataLoader(path=dataset_path)
builder = AcqusitionBuilder(dataset=loader.dataset)
builder

## Inspect an individual stream

`get_reward_delivery` returns the filtered `OutputSet` write messages from the Behavior Board as a `pandas.DataFrame`. Use this when you want the raw frame rather than the typed acquisition object.

In [3]:
reward_writes = builder.get_reward_delivery()
reward_writes.head()

,DOPort0,DOPort1,DOPort2,SupplyPort0,SupplyPort1,SupplyPort2,Led0,Led1,Rgb0,Rgb1,DO0,DO1,DO2,DO3,MessageType
Time,,,,,,,,,,,,,,,
2.504232e+06,False,False,False,True,False,False,False,False,False,False,False,False,False,False,WRITE
2.504279e+06,False,False,False,True,False,False,False,False,False,False,False,False,False,False,WRITE
2.504341e+06,False,False,False,True,False,False,False,False,False,False,False,False,False,False,WRITE
2.504399e+06,False,False,False,True,False,False,False,False,False,False,False,False,False,False,WRITE
2.504468e+06,False,False,False,True,False,False,False,False,False,False,False,False,False,False,WRITE


## Build the Acquisition Collection object

`build_acquisition` returns an `AcquisitionCollection` pydantic model containing one `AcquisitionSeries` per stream. Each series holds `data`, `timestamps`, `unit`, and `description` fields ready to be mapped to a `pynwb.TimeSeries` by a downstream writer.

In [4]:
acquisition = builder.build_acquisition()
acquisition

AcquisitionCollection(left_reward_delivery_time=AcquisitionSeries(name='left_reward_delivery_time', data=array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False,  True,  True,  True,  True,  True,  True, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False,  True,  True,  True,  True,  True,  True, False,
        True,  True,  True,  True,  True, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False,  True, False,  True,  True, False,
        True,  True,  True,  True, False, False, False, False, False,
       False, False, False,  True,  True,  True,  True,

In [5]:
left = acquisition.left_reward_delivery_time
print(f"name:        {left.name}")
print(f"unit:        {left.unit}")
print(f"description: {left.description}")
print(f"n samples:   {left.data.shape[0]}")
left.timestamps[:10]

name:        left_reward_delivery_time
unit:        second
description: The reward delivery time of the left lick port. The data field annotates whether the reward was earned, manual, or automatic
n samples:   207


array([2504232.053504, 2504279.20848 , 2504341.438496, 2504398.506496,
       2504467.84448 , 2504514.383488, 2504565.286496, 2504618.533504,
       2504618.881504, 2504627.669504])

## Serialize for downstream use

Because `AcquisitionCollection` is a pydantic model, you can dump the field structure (without the heavy array payloads) for logging or manifest creation.

In [6]:
{
    name: {
        "unit": series.unit,
        "description": series.description,
        "n_samples": series.data.shape[0],
    }
    for name, series in acquisition
}

{'left_reward_delivery_time': {'unit': 'second',
  'description': 'The reward delivery time of the left lick port. The data field annotates whether the reward was earned, manual, or automatic',
  'n_samples': 207},
 'right_reward_delivery_time': {'unit': 'second',
  'description': 'The reward delivery time of the right lick port. The data field annotates whether the reward was earned, manual, or automatic',
  'n_samples': 207}}